# CN7+RG3 RH 가스 공동학습
제품을 통합하되 캐비티는 RH 전용 유지. 제품별 성능과 단독학습 대비 변화 확인.

In [1]:
from pathlib import Path
import json, subprocess
import numpy as np,pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
tables=root/'reports/tables'
shot=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
keys=['EQUIP_CD','product','TimeStamp']
parts=[]
for p in ['CN7','RG3']:
    raw=pd.read_csv(root/('data/original_split/moldset_labeled_'+p.lower()+'.csv'))
    raw['product']=p;raw['TimeStamp']=pd.to_datetime(raw.TimeStamp)
    raw['side']=raw.PART_NAME.str.extract(r'(LH|RH)\s*$',expand=False)
    raw['Reason']=raw.Reason.fillna('정상')
    raw=raw.loc[raw.side.eq('RH')]
    if p=='CN7':raw=raw.loc[~raw.PART_FACT_PLAN_DATE.str.startswith('2020-10-27')]
    raw=raw.loc[raw.PassOrFail.eq(0)|raw.Reason.eq('가스')]
    parts.append(raw[keys+['Reason','PassOrFail']])
target=pd.concat(parts,ignore_index=True)
assert not target.duplicated(keys).any()
d=target.merge(shot.drop(columns=['Reason','PassOrFail']),on=keys,validate='one_to_one').sort_values(['product','TimeStamp']).reset_index(drop=True)
d['is_RG3']=d['product'].eq('RG3').astype(int)
y=d.PassOrFail.to_numpy(int);groups=d.comparison_block
counts=d.groupby('product').agg(shots=('PassOrFail','size'),gas=('PassOrFail','sum'))
counts['normal']=counts.shots-counts.gas
assert counts.loc['RG3','shots']==583 and counts.loc['RG3','gas']==17 and counts.loc['CN7','gas']==10
counts.to_csv(tables/'gas_product_pooling_rh_counts.csv',encoding='utf-8-sig')
stdcols=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
relations={
'pressure_residual':('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time']),
'plasticizing_residual':('Plasticizing_Time',['Average_Screw_RPM','Average_Back_Pressure']),
'cushion_residual':('Cushion_Position',['Plasticizing_Position','Max_Injection_Pressure']),
'mold_residual':('Mold_Temperature_4',['Mold_Temperature_3','Barrel_Temperature_5'])}
rcache={}
def residuals(ref,query):
    key=(tuple(ref),tuple(query))
    if key in rcache:return rcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[query])
    result=np.full((len(query),4),np.nan)
    for p in ['CN7','RG3']:
        nm=d.iloc[ref].loc[(y[ref]==0)&d.iloc[ref]['product'].eq(p).to_numpy()]
        mask=d.iloc[query]['product'].eq(p).to_numpy()
        if len(nm)<10 or not mask.any():continue
        q=d.iloc[query].loc[mask]
        for j,(t,fs) in enumerate(relations.values()):
            sc=RobustScaler().fit(nm[fs]);m=Ridge(alpha=10).fit(sc.transform(nm[fs]),nm[t])
            fitted=m.predict(sc.transform(nm[fs]))
            scale=max(np.sqrt(np.mean((nm[t].to_numpy()-fitted)**2)),float(nm[t].std(ddof=0))*.1,1e-6)
            result[mask,j]=(q[t].to_numpy()-m.predict(sc.transform(q[fs])))/scale
    rcache[key]=result
    return result
def rf(x,yy,q):
    if len(np.unique(yy))<2:return np.full(len(q),float(yy[0]) if len(yy) else 0.)
    model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
    return model.fit(x,yy).predict_proba(q)[:,1]
pcache={}
def pair(train,test,product_id):
    key=(tuple(train),tuple(test),product_id)
    if key in pcache:return pcache[key]
    fs=stdcols+(['is_RG3'] if product_id else [])
    a=rf(d.iloc[train][fs],y[train],d.iloc[test][fs])
    tr_res=np.full((len(train),4),np.nan)
    for b in groups.iloc[train].unique():
        mask=groups.iloc[train].eq(b).to_numpy()
        if (~mask).any():tr_res[mask]=residuals(train[~mask],train[mask])
    te_res=residuals(train,test) if len(train) else np.full((len(test),4),np.nan)
    if product_id:
        tr_res=np.column_stack([tr_res,d.is_RG3.iloc[train]])
        te_res=np.column_stack([te_res,d.is_RG3.iloc[test]])
    b=rf(tr_res,y[train],te_res)
    pcache[key]=np.column_stack([a,b])
    return pcache[key]
records=[];logs=[]
for setting,pooled,product_id in [('separate_products',False,False),('pooled_with_product',True,True),('pooled_no_product',True,False)]:
    for block in groups.unique():
        test=np.flatnonzero(groups.eq(block));p=d['product'].iloc[test[0]]
        trmask=~groups.eq(block)
        if not pooled:trmask &= d['product'].eq(p)
        train=np.flatnonzero(trmask)
        outer=pair(train,test,product_id)
        inner=np.full((len(train),2),np.nan)
        for ib in groups.iloc[train].unique():
            mask=groups.iloc[train].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=pair(train[~mask],train[mask],product_id)
        for name,k in [('std_rf',0),('residual_rf',1),('max_union',None)]:
            s=outer[:,k] if k is not None else outer.max(axis=1)
            cs=inner[:,k] if k is not None else inner.max(axis=1)
            normal=(y[train]==0)&d['product'].iloc[train].eq(p).to_numpy()&np.isfinite(cs)
            cal=cs[normal]
            threshold=float(np.quantile(cal,.99,method='higher')) if len(cal)>=10 else np.nan
            logs.append(dict(setting=setting,model=name,block=block,product=p,train=len(train),train_gas=int(y[train].sum()),test=len(test),test_gas=int(y[test].sum()),threshold=threshold,normal_calibration_n=len(cal)))
            for j,i in enumerate(test):
                records.append(dict(setting=setting,model=name,product=p,TimeStamp=d.TimeStamp.iloc[i],comparison_block=block,actual=int(y[i]),score=float(s[j]),threshold=threshold,pred05=bool(s[j]>=.5),pred_calibrated=bool(s[j]>threshold) if np.isfinite(threshold) else np.nan))
        print('Completed',setting,block,flush=True)
pred=pd.DataFrame(records);metrics=[]
for (setting,name,p),g in pred.groupby(['setting','model','product'],sort=False):
    for rule,col in [('fixed05','pred05'),('train_fpr1','pred_calibrated')]:
        good=g[col].notna();gg=g.loc[good].reset_index(drop=True)
        if not len(gg):
            metrics.append(dict(setting=setting,model=name,product=p,rule=rule,shots=0,gas=0,status='insufficient nested calibration',TP=np.nan,FP=np.nan,FN=np.nan,TN=np.nan,AP=np.nan,top10_TP=np.nan,top10_selected=np.nan));continue
        selected=np.zeros(len(gg),bool)
        for _,part in gg.groupby('comparison_block',sort=False):
            ids=part.index.to_numpy();tie=np.random.RandomState(42).rand(len(ids))
            selected[ids[np.lexsort((tie,-part.score.to_numpy()))[:int(np.ceil(len(ids)*.1))]]]=True
        tn,fp,fn,tp=confusion_matrix(gg.actual,gg[col].astype(bool),labels=[0,1]).ravel()
        metrics.append(dict(setting=setting,model=name,product=p,rule=rule,shots=len(gg),gas=int(gg.actual.sum()),status='evaluated',TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(gg.actual,gg.score),top10_TP=int(gg.actual.to_numpy()[selected].sum()),top10_selected=int(selected.sum())))
mt=pd.DataFrame(metrics)
mt.to_csv(tables/'gas_product_pooling_metrics.csv',index=False,encoding='utf-8-sig')
pred.to_csv(tables/'gas_product_pooling_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(logs).to_csv(tables/'gas_product_pooling_fold_audit.csv',index=False,encoding='utf-8-sig')
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
for name,oldname in [('std_rf','std4'),('residual_rf','residual4'),('max_union','max_union')]:
    a=pred.loc[pred.setting.eq('separate_products')&pred['product'].eq('RG3')&pred.model.eq(name)]
    b=old.loc[old.scheme.eq('leave_one_block_out')&old.model.eq(oldname)]
    chk=a.merge(b[['TimeStamp','score','threshold']],on='TimeStamp',suffixes=('_new','_old'),validate='one_to_one')
    assert len(chk)==583 and np.allclose(chk.score_new,chk.score_old) and np.allclose(chk.threshold_new,chk.threshold_old)
lines=['# CN7+RG3 RH 가스 공동학습 비교','',
'## 대상','',counts.to_string(),'',
'- 이전 AP0.1686/TP4/FP6은 RG3 가스 실험이었다. 이번에는 CN7을 추가했다.',
'- 대회 범위 CN7에서 RH 가스10개만 포함. LH 가스3개는 RH 전용 대상이 아니므로 포함하지 않음. CN7 10/27 초기허용 기간 전체와 양제품 미성형 제외.',
'- LH/RH를 합치지 않고 raw RH 라벨을 사용. CN7 RH 정상에 LH만 불량인 Shot이 포함될 수 있으나 RH 정상 정답이므로 유지.',
'- 제품별로 따로학습 vs 공동학습+제품ID vs 공동학습 ID없음 비교. 제품ID는5번째입력이며 관계잔차는제품별정상에서따로학습.',
'- 같은RF200/depth8/leaf3/seed42, 같은구간전체제외. 잔차RF 학습변수는안쪽구간제외, 판정99% 기준도제품별학습정상안쪽예측만 사용.',
'- CN7은대회범위관측구간이2개로매우적다. 별도학습의바깥학습에1구간만남으면안쪽잔차/기준보정이부족하다. 이때결측과미평가를명시하고통합과확정비교하지않음.',
'- 기존RG3단독점수와기준은파일대조하여정확히재현검증.',
'- 미래학습구간이포함될수있는구간제외평가이며미래생산검증아님. 반복탐색후독립성한계가있음.',
'- 평균RPM관련잔차는정의불확실성을남겨둔기존모델재현용. 물리원인으로해석하지않음.',
'','## 제품별 결과','',
'| setting | model | product | rule | shots | gas | TP | FP | FN | TN | AP | top10_TP | top10_selected |',
'|---|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for r in mt.itertuples():lines.append(f'| {r.setting} | {r.model} | {r.product} | {r.rule} | {r.shots} | {r.gas} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} | {r.top10_TP} | {r.top10_selected} |')
lines+=['','AP를제품통합숫자하나로만보고하면CN7의쉬운사례가RG3향상처럼보일수있어제품별로보고한다. CN7 RH 대상과이전CN7 Shot합산대상의라벨수가달라그수치를직접비교하면안된다.','',
'산출물: notebooks/23_cn7_rg3_rh_gas_joint_learning.ipynb, reports/tables/gas_product_pooling_metrics.csv 및 predictions/fold_audit/counts.']
(root/'reports/cn7_rg3_rh_gas_joint_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(counts.to_string())
print(mt.loc[mt.model.eq('max_union')].to_string(index=False))


Completed separate_products 20201016_045747_CN7
Completed separate_products 20201019_232446_CN7
Completed separate_products 20201021_001626_RG3
Completed separate_products 20201022_002405_RG3
Completed separate_products 20201022_044026_RG3
Completed separate_products 20201023_010128_RG3
Completed separate_products 20201023_044148_RG3
Completed pooled_with_product 20201016_045747_CN7
Completed pooled_with_product 20201019_232446_CN7
Completed pooled_with_product 20201021_001626_RG3
Completed pooled_with_product 20201022_002405_RG3
Completed pooled_with_product 20201022_044026_RG3
Completed pooled_with_product 20201023_010128_RG3
Completed pooled_with_product 20201023_044148_RG3
Completed pooled_no_product 20201016_045747_CN7
Completed pooled_no_product 20201019_232446_CN7
Completed pooled_no_product 20201021_001626_RG3
Completed pooled_no_product 20201022_002405_RG3
Completed pooled_no_product 20201022_044026_RG3
Completed pooled_no_product 20201023_010128_RG3
Completed pooled_no_produc